# DFT From Scratch — Notebook 01
## Schrödinger Equation → Finite-Difference Hamiltonian → Eigenvalues/Eigenvectors

This notebook is the first numerical building block of a from-scratch DFT code.

We will **not** use a DFT package. We will construct the quantum-mechanical Hamiltonian ourselves:

\[
\hat H =
-\frac{\hbar^2}{2m}\frac{d^2}{dx^2}+V(x)
\]

and solve

\[
\hat H\psi_n(x)=E_n\psi_n(x).
\]

### What we will learn

1. Discretize a continuous coordinate \(x\) onto a grid.
2. Derive the finite-difference approximation to \(d^2/dx^2\).
3. Construct the kinetic-energy matrix.
4. Add an external potential \(V(x)\).
5. Diagonalize the Hamiltonian.
6. Normalize and visualize the eigenfunctions.
7. Compare numerical energies with an analytical infinite square well.
8. Study a harmonic oscillator.
9. Understand convergence with grid spacing.

This notebook is deliberately written so that the same structure can later be translated to **Fortran**.

## 1. The time-independent Schrödinger equation

For one particle in one spatial dimension,

\[
\left[
-\frac{\hbar^2}{2m}\frac{d^2}{dx^2}+V(x)
\right]\psi(x)
=
E\psi(x).
\]

The two pieces are

\[
\hat T=-\frac{\hbar^2}{2m}\frac{d^2}{dx^2},
\qquad
\hat V=V(x).
\]

Therefore,

\[
\hat H=\hat T+\hat V.
\]

The DFT code we eventually build will retain this basic structure. The major difference is that DFT replaces the externally specified potential by an effective Kohn–Sham potential,

\[
V_{\rm KS}(\mathbf r)
=
V_{\rm ext}(\mathbf r)
+
V_H(\mathbf r)
+
V_{xc}(\mathbf r).
\]

So this notebook is the **single-particle numerical foundation**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility / numerical display
np.set_printoptions(precision=6, suppress=True)

print("NumPy version:", np.__version__)

## 2. Discretizing space

Choose a finite interval

\[
x\in[0,L]
\]

and divide it into \(N\) equally spaced points:

\[
x_j=j\Delta x.
\]

The grid spacing is

\[
\Delta x=\frac{L}{N-1}.
\]

A continuous wavefunction

\[
\psi(x)
\]

becomes a vector

\[
\boldsymbol\psi=
(\psi_0,\psi_1,\ldots,\psi_{N-1})^T.
\]

The differential equation can therefore be converted into a matrix eigenvalue problem.

In [ ]:
# Basic grid
L = 1.0
N = 200

x = np.linspace(0.0, L, N)
dx = x[1] - x[0]

print("L  =", L)
print("N  =", N)
print("dx =", dx)

## 3. Finite-difference approximation to the second derivative

Expand \(\psi(x+\Delta x)\) and \(\psi(x-\Delta x)\) in Taylor series:

\[
\psi(x+\Delta x)
=
\psi(x)+\Delta x\,\psi'(x)
+\frac{\Delta x^2}{2}\psi''(x)+\cdots
\]

and

\[
\psi(x-\Delta x)
=
\psi(x)-\Delta x\,\psi'(x)
+\frac{\Delta x^2}{2}\psi''(x)+\cdots.
\]

Adding them,

\[
\psi(x+\Delta x)+\psi(x-\Delta x)
=
2\psi(x)+\Delta x^2\psi''(x)+O(\Delta x^4).
\]

Therefore,

\[
\boxed{
\psi''(x)
\approx
\frac{\psi_{j+1}-2\psi_j+\psi_{j-1}}
{\Delta x^2}
}
\]

with an error of order \(O(\Delta x^2)\).

Substituting into the kinetic-energy operator,

\[
-\frac{\hbar^2}{2m}\psi''(x)
\approx
-\frac{\hbar^2}{2m\Delta x^2}
\left(
\psi_{j+1}-2\psi_j+\psi_{j-1}
\right).
\]

Thus the kinetic-energy matrix is tridiagonal.

## 4. Use dimensionless atomic-style units

To make the first notebook simple, we choose units such that

\[
\hbar=1,\qquad m=1.
\]

Then

\[
\hat T=-\frac12\frac{d^2}{dx^2}.
\]

The finite-difference kinetic matrix has

\[
T_{jj}=\frac{1}{\Delta x^2}
\]

and

\[
T_{j,j\pm1}=-\frac{1}{2\Delta x^2}.
\]

This is exactly the matrix we will construct below.

In [ ]:
# Atomic-style dimensionless units
hbar = 1.0
mass = 1.0

# Kinetic-energy matrix
T = np.zeros((N, N))

for j in range(1, N-1):
    T[j, j]     = hbar**2 / (mass * dx**2)
    T[j, j-1]   = -hbar**2 / (2 * mass * dx**2)
    T[j, j+1]   = -hbar**2 / (2 * mass * dx**2)

print("Kinetic matrix shape:", T.shape)
print("Central diagonal element:", T[N//2, N//2])

### Why the boundary rows are zero here

We are going to impose **Dirichlet boundary conditions**

\[
\psi(0)=\psi(L)=0.
\]

The boundary values are therefore fixed rather than treated as unknown dynamical amplitudes.

For a cleaner numerical implementation, we can solve only for the \(N-2\) interior grid points.

In [ ]:
# Interior grid points only: j = 1,...,N-2
x_in = x[1:-1]
M = len(x_in)

T_in = np.zeros((M, M))

for j in range(M):
    T_in[j, j] = hbar**2 / (mass * dx**2)
    if j > 0:
        T_in[j, j-1] = -hbar**2 / (2 * mass * dx**2)
    if j < M-1:
        T_in[j, j+1] = -hbar**2 / (2 * mass * dx**2)

print("Interior Hamiltonian dimension:", M, "x", M)

# Show a small representative matrix
small_M = 8
print(T_in[:small_M, :small_M])

## 5. Add an external potential

For the first physical test we use the infinite square well.

Inside the box,

\[
V(x)=0,
\]

while the wavefunction obeys

\[
\psi(0)=\psi(L)=0.
\]

Because the boundaries are enforced explicitly, we do not need to put an enormous potential at the endpoints.

Therefore,

\[
H=T+V=T.
\]

Later we will replace \(V(x)\) with a harmonic oscillator and other potentials.

In [ ]:
# Infinite square well: V(x) = 0 inside the domain
V_in = np.zeros(M)

H = T_in + np.diag(V_in)

# Check Hermiticity
hermiticity_error = np.max(np.abs(H - H.T.conj()))

print("Hermiticity error =", hermiticity_error)

## 6. Solve the matrix eigenvalue problem

We now have

\[
H\boldsymbol\psi_n=E_n\boldsymbol\psi_n.
\]

Because \(H\) is real symmetric (Hermitian), we use

```python
np.linalg.eigh(H)
```

which returns eigenvalues in ascending order and orthonormal eigenvectors.

This is the first place where a numerical linear-algebra routine enters the calculation. Everything before this point — the physical operator and its discretization — was constructed explicitly.

In [ ]:
energies, states = np.linalg.eigh(H)

print("Lowest 10 numerical eigenvalues:")
for n, E in enumerate(energies[:10], start=1):
    print(f"n={n:2d}   E={E:.10f}")

## 7. Analytical solution for the infinite square well

For an infinite square well of width \(L\),

\[
\psi_n(x)
=
\sqrt{\frac{2}{L}}
\sin\left(\frac{n\pi x}{L}\right),
\]

and

\[
\boxed{
E_n=
\frac{\hbar^2\pi^2n^2}{2mL^2}
}
\]

in our units \(\hbar=m=1\):

\[
E_n=\frac{\pi^2n^2}{2L^2}.
\]

This gives us an exact benchmark for our finite-difference calculation.

In [ ]:
n_levels = 8
n_values = np.arange(1, n_levels + 1)

E_exact = (hbar**2 * np.pi**2 * n_values**2) / (2 * mass * L**2)
E_numeric = energies[:n_levels]

print(" n       numerical          exact          relative error")
print("-" * 62)

for n, En, Ee in zip(n_values, E_numeric, E_exact):
    err = abs((En - Ee) / Ee)
    print(f"{n:2d}   {En:16.10f}   {Ee:16.10f}   {err:12.4e}")

In [ ]:
# Plot numerical vs analytical energies
plt.figure(figsize=(7, 5))
plt.plot(n_values, E_numeric, "o-", label="Finite difference")
plt.plot(n_values, E_exact, "s--", label="Analytical")
plt.xlabel("Quantum number n")
plt.ylabel("Energy")
plt.title("Infinite Square Well: Numerical vs Analytical")
plt.legend()
plt.grid(True)
plt.show()

## 8. Normalize the eigenvectors using the physical integral

`np.linalg.eigh` normalizes the eigenvectors according to the ordinary vector norm,

\[
\sum_j |\psi_j|^2=1.
\]

But the continuous normalization condition is

\[
\int |\psi(x)|^2\,dx=1.
\]

For a grid this becomes approximately

\[
\sum_j |\psi_j|^2\Delta x=1.
\]

Therefore we explicitly renormalize each state using the grid spacing.

In [ ]:
def normalize_wavefunction(psi, dx):
    norm = np.sqrt(np.sum(np.abs(psi)**2) * dx)
    return psi / norm

states_phys = np.zeros_like(states)

for n in range(states.shape[1]):
    states_phys[:, n] = normalize_wavefunction(states[:, n], dx)

for n in range(3):
    norm = np.sum(np.abs(states_phys[:, n])**2) * dx
    print(f"State {n+1}: integral |psi|^2 dx = {norm:.12f}")

## 9. Compare numerical and analytical wavefunctions

The sign of an eigenvector is arbitrary:

\[
\psi(x)\rightarrow-\psi(x)
\]

describes exactly the same quantum state.

Therefore, when comparing a numerical eigenfunction with an analytical one, we may need to flip the sign.

In [ ]:
plt.figure(figsize=(9, 6))

for n in range(3):
    psi_num = states_phys[:, n]

    psi_exact = np.sqrt(2/L) * np.sin((n+1)*np.pi*x_in/L)

    # Fix arbitrary eigenvector sign
    overlap = np.sum(psi_num * psi_exact) * dx
    if overlap < 0:
        psi_num = -psi_num

    plt.plot(x_in, psi_num, label=f"Numerical n={n+1}")
    plt.plot(x_in, psi_exact, "--", label=f"Exact n={n+1}")

plt.xlabel("x")
plt.ylabel(r"$\psi_n(x)$")
plt.title("Infinite Square Well Eigenfunctions")
plt.legend()
plt.grid(True)
plt.show()

## 10. Probability density

The physically observable position probability density is

\[
\rho_n(x)=|\psi_n(x)|^2.
\]

For a normalized state,

\[
\int_0^L \rho_n(x)\,dx=1.
\]

In [ ]:
plt.figure(figsize=(9, 6))

for n in range(3):
    density = np.abs(states_phys[:, n])**2
    plt.plot(x_in, density, label=f"n={n+1}")

plt.xlabel("x")
plt.ylabel(r"$|\psi_n(x)|^2$")
plt.title("Probability Densities")
plt.legend()
plt.grid(True)
plt.show()

# 11. A more general potential

The whole method becomes useful once we allow

\[
V(x)\neq0.
\]

The potential is represented by a diagonal matrix:

\[
V_{ij}=V(x_i)\delta_{ij}.
\]

Then

\[
H=T+V.
\]

Let's test this with a harmonic oscillator,

\[
V(x)=\frac12m\omega^2x^2.
\]

For \(\hbar=m=\omega=1\),

\[
E_n=n+\frac12,
\qquad n=0,1,2,\ldots
\]

analytically.

In [ ]:
# Harmonic oscillator on a symmetric finite domain
L_ho = 10.0
N_ho = 800

x_ho = np.linspace(-L_ho/2, L_ho/2, N_ho)
dx_ho = x_ho[1] - x_ho[0]
xhi = x_ho[1:-1]
M_ho = len(xhi)

omega = 1.0
mass_ho = 1.0
hbar_ho = 1.0

T_ho = np.zeros((M_ho, M_ho))

for j in range(M_ho):
    T_ho[j, j] = hbar_ho**2 / (mass_ho * dx_ho**2)
    if j > 0:
        T_ho[j, j-1] = -hbar_ho**2 / (2*mass_ho*dx_ho**2)
    if j < M_ho-1:
        T_ho[j, j+1] = -hbar_ho**2 / (2*mass_ho*dx_ho**2)

V_ho = 0.5 * mass_ho * omega**2 * xhi**2
H_ho = T_ho + np.diag(V_ho)

E_ho, psi_ho = np.linalg.eigh(H_ho)

print("Harmonic oscillator:")
for n in range(6):
    exact = hbar_ho * omega * (n + 0.5)
    print(f"n={n:2d}: numerical={E_ho[n]:.8f}, exact={exact:.8f}, error={E_ho[n]-exact:+.3e}")

In [ ]:
# Plot harmonic oscillator potential and first few probability densities
plt.figure(figsize=(9, 6))

scale = 0.8

for n in range(4):
    psi = normalize_wavefunction(psi_ho[:, n], dx_ho)
    density = np.abs(psi)**2
    plt.plot(xhi, E_ho[n] + scale*density, label=f"n={n}")

plt.plot(xhi, V_ho, "k--", label="V(x)")
plt.xlabel("x")
plt.ylabel("Energy / shifted probability density")
plt.title("Harmonic Oscillator: Eigenstates")
plt.legend()
plt.grid(True)
plt.show()

# 12. Convergence with grid spacing

The finite-difference approximation is second-order:

\[
\text{error}\sim O(\Delta x^2).
\]

Therefore, as we decrease \(\Delta x\), the numerical energy should converge toward the exact result.

We test the ground-state energy of the infinite square well for several grid sizes.

In [ ]:
def square_well_ground_energy(N, L=1.0):
    x = np.linspace(0, L, N)
    dx = x[1] - x[0]
    M = N - 2

    diag = np.full(M, hbar**2/(mass*dx**2))
    off  = np.full(M-1, -hbar**2/(2*mass*dx**2))

    H = np.diag(diag) + np.diag(off, 1) + np.diag(off, -1)

    eigvals = np.linalg.eigvalsh(H)
    return eigvals[0], dx

Ns = [50, 100, 200, 400, 800]
results = []

E0_exact = np.pi**2 / 2

for Ntest in Ns:
    E0, dxtest = square_well_ground_energy(Ntest)
    error = abs(E0 - E0_exact)
    results.append((Ntest, dxtest, E0, error))

print("N       dx              E0                 absolute error")
print("-"*65)
for row in results:
    print(f"{row[0]:4d}   {row[1]:.8f}   {row[2]:.12f}   {row[3]:.4e}")

In [ ]:
dxs = np.array([r[1] for r in results])
errs = np.array([r[3] for r in results])

plt.figure(figsize=(7, 5))
plt.loglog(dxs, errs, "o-")
plt.xlabel(r"$\Delta x$")
plt.ylabel("Absolute ground-state energy error")
plt.title("Finite-Difference Convergence")
plt.grid(True, which="both")
plt.show()

# 13. Inspect the Hamiltonian structure

For a constant potential, the Hamiltonian has the form

\[
H=
\begin{pmatrix}
\frac{1}{\Delta x^2}+V_1&
-\frac{1}{2\Delta x^2}&0&\cdots\\
-\frac{1}{2\Delta x^2}&
\frac{1}{\Delta x^2}+V_2&
-\frac{1}{2\Delta x^2}&\cdots\\
0&-\frac{1}{2\Delta x^2}&
\frac{1}{\Delta x^2}+V_3&\cdots\\
\vdots&\vdots&\vdots&\ddots
\end{pmatrix}.
\]

This **sparse tridiagonal structure** is extremely important.

For a realistic 3-D DFT calculation, the Hamiltonian can contain millions of degrees of freedom, so storing a dense matrix is generally impossible. Later notebooks will therefore introduce sparse matrices and iterative eigensolvers.

In [ ]:
# Visualize a small Hamiltonian matrix
M_show = 40

plt.figure(figsize=(6, 5))
plt.spy(H[:M_show, :M_show], markersize=4)
plt.xlabel("Column")
plt.ylabel("Row")
plt.title("Sparsity Pattern of the Finite-Difference Hamiltonian")
plt.show()

# 14. What this notebook has actually accomplished

We started with the differential equation

\[
\boxed{
\left[-\frac{\hbar^2}{2m}\frac{d^2}{dx^2}+V(x)\right]\psi=E\psi
}
\]

and converted it into

\[
\boxed{
H\mathbf\psi=E\mathbf\psi
}
\]

by discretizing space.

The computational chain is

\[
\boxed{
x
\rightarrow
\Delta x
\rightarrow
\frac{d^2}{dx^2}
\rightarrow
T
\rightarrow
V
\rightarrow
H
\rightarrow
(E_n,\psi_n)
}
\]

This is the first layer of our future DFT implementation.

# 15. Connection to Kohn–Sham DFT

In Kohn–Sham DFT we will eventually solve

\[
\left[
-\frac{\hbar^2}{2m}\nabla^2
+
V_{\rm KS}(\mathbf r)
\right]
\psi_i(\mathbf r)
=
\epsilon_i\psi_i(\mathbf r),
\]

where

\[
V_{\rm KS}
=
V_{\rm ext}+V_H+V_{xc}.
\]

Compare this directly with today's calculation:

| Notebook 01 | Future DFT |
|---|---|
| \(x\) | \(\mathbf r=(x,y,z)\) |
| \(d^2/dx^2\) | \(\nabla^2\) |
| \(V(x)\) | \(V_{\rm KS}(\mathbf r)\) |
| one-particle \(\psi_n\) | Kohn–Sham orbitals \(\psi_i\) |
| \(E_n\) | \(\epsilon_i\) |
| one diagonalization | repeated SCF diagonalizations |

The crucial new ingredient in DFT is that

\[
V_{\rm KS}=V_{\rm KS}[n]
\]

depends on the electron density, and the density itself depends on the orbitals:

\[
n(\mathbf r)
=
\sum_i f_i|\psi_i(\mathbf r)|^2.
\]

That feedback loop is what Notebook 08 will eventually implement.

# 16. Exercises

Try these before moving to Notebook 02.

### Exercise 1
Change \(N\) from 200 to 50, 100, 400 and compare the ground-state energy.

### Exercise 2
Replace the square-well potential by

\[
V(x)=V_0\sin^2\left(\frac{2\pi x}{L}\right).
\]

Plot the first five eigenstates.

### Exercise 3
Make a double-well potential:

\[
V(x)=a(x^2-b^2)^2.
\]

Investigate how the lowest two energies approach one another as the wells become farther apart.

### Exercise 4
Replace `np.linalg.eigh` with a sparse eigensolver in a later experiment.

### Exercise 5
Rewrite the kinetic-energy matrix using only the three diagonals rather than nested Python loops.

These exercises directly prepare the numerical ideas needed for periodic DFT.

# 17. Roadmap

The next notebook will introduce **periodicity**.

### Notebook 02 — Periodic Boundary Conditions

We will implement

\[
\psi(x+L)=e^{ikL}\psi(x),
\]

introduce Bloch's theorem,

\[
\psi_{nk}(x)=e^{ikx}u_{nk}(x),
\]

and construct the Hamiltonian separately for different \(k\)-points.

That will take us from a simple quantum particle to the beginning of **band-structure calculations**.

The later sequence will then build toward:

\[
\text{finite difference}
\rightarrow
\text{periodic crystal}
\rightarrow
k\text{-points}
\rightarrow
\text{electron density}
\rightarrow
V_H
\rightarrow
V_{xc}
\rightarrow
\boxed{\text{Kohn--Sham SCF}}
\rightarrow
\text{bands/DOS}
\rightarrow
\text{spin-DFT}
\rightarrow
\text{SOC}
\rightarrow
\text{Berry/Wannier physics}.
\]